# 02 - Baseline Models

dos modelos baseline para aprobación de créditos:

1. Regresión Logística
2. Random Forest



In [9]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root added to sys.path:', project_root)

Project root added to sys.path: C:\Users\paula\mlops-proyecto-final-equipo8


In [10]:
from pathlib import Path
import pandas as pd

from src.data import load_dataset, apply_basic_cleaning, apply_quality_fixes, build_data_quality_report
from src.features import add_financial_features
from src.models import train_baseline_models

In [11]:
DATA_PATH = Path('../data/loan_approval_dataset.csv')

df = load_dataset(DATA_PATH)
df = apply_basic_cleaning(df)
quality_report = build_data_quality_report(df)
quality_report

{'rows': 4269,
 'columns': 13,
 'duplicate_rows': 0,
 'null_counts': {'loan_id': 0,
  'no_of_dependents': 0,
  'education': 0,
  'self_employed': 0,
  'income_annum': 0,
  'loan_amount': 0,
  'loan_term': 0,
  'cibil_score': 0,
  'residential_assets_value': 0,
  'commercial_assets_value': 0,
  'luxury_assets_value': 0,
  'bank_asset_value': 0,
  'loan_status': 0},
 'negative_counts': {'income_annum': 0,
  'loan_amount': 0,
  'residential_assets_value': 28,
  'commercial_assets_value': 0,
  'luxury_assets_value': 0,
  'bank_asset_value': 0}}

In [12]:
df = apply_quality_fixes(df)
df = add_financial_features(df)

print('Shape con features:', df.shape)
df.head()

Shape con features: (4269, 17)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,total_assets,loan_to_income_ratio,loan_to_assets_ratio,net_worth_proxy
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved,50700000,3.114583,0.589744,20800000
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected,17000000,2.975610,0.717647,4800000
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected,57700000,3.263736,0.514731,28000000
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected,52700000,3.743902,0.582543,22000000
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected,55000000,2.469388,0.440000,30800000


In [13]:
results = train_baseline_models(df)
results['metrics']

{'logistic_regression': {'accuracy': 0.9180327868852459,
  'f1': 0.933206106870229,
  'recall': 0.9209039548022598,
  'roc_auc': 0.9733198066618857},
 'random_forest': {'accuracy': 0.9988290398126464,
  'f1': 0.9990592662276576,
  'recall': 1.0,
  'roc_auc': 1.0}}

In [14]:
metrics_df = pd.DataFrame(results['metrics']).T.sort_values('f1', ascending=False)
metrics_df

,accuracy,f1,recall,roc_auc
random_forest,0.998829,0.999059,1.000000,1.00000
logistic_regression,0.918033,0.933206,0.920904,0.97332


## Conclusión

En este baseline se validó un flujo de modelado para aprobación de créditos: carga de datos, limpieza, control de calidad, corrección de anomalías y creación de variables financieras derivadas.

Los principales hallazgos fueron:

1. El dataset quedó en condiciones adecuadas para modelado después de aplicar reglas de calidad (por ejemplo, corrección de valores negativos en activos residenciales).
2. El feature engineering aportó variables (`total_assets`, `loan_to_income_ratio`, `loan_to_assets_ratio`, `net_worth_proxy`).
3. En la comparación de modelos, Random Forest superó a Regresión Logística en las métricas evaluadas (`accuracy`, `f1`, `recall`, `roc_auc`), por lo que es el mejor para la siguiente fase.
